In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
off_path = '/kaggle/input/datasets/mansiaggarwal88/open-food-facts-2025-1-35m-global-food-dataset/open_food_facts_2025 (1).csv'
off = pd.read_csv(off_path, dtype=str)
print(off.shape)
print(off.isna().mean().round(3))

In [ ]:
nutrients = ['proteins_100g','fat_100g','carbohydrates_100g','sugars_100g','fiber_100g', 'saturated-fat_100g','sodium_100g','salt_100g']

for nutr in nutrients:
    off[nutr] = pd.to_numeric(off[nutr])

has_so =off['sodium_100g'].notna()
has_sa = off['salt_100g'].notna()
# checking if both sodium and salt are filled, if only sodium not filled but salt is filled use formula to fill it
print('number of sodium filled:', has_so.sum())
print('both filled:', (has_so & has_sa).sum())
print('only salt',(~has_so & has_sa).sum())

In [ ]:
nutr7 = ['proteins_100g','fat_100g','carbohydrates_100g','sugars_100g','fiber_100g', 'saturated-fat_100g','sodium_100g']
keep = ['code','product_name','ingredients_text','image_url', 'countries_en','nova_group','categories_en', 'pnns_groups_2','energy-kcal_100g'] + nutr7
off['energy-kcal_100g'] = pd.to_numeric(off['energy-kcal_100g'])
off_selected_columns = off[keep].copy()
print(off_selected_columns.shape)

In [ ]:
# check for missing value and black text field
for c in ['product_name','ingredients_text','image_url']:
    s= off_selected_columns[c]
    n_nan = s.isna().sum()
    n_blank = (s.str.strip()=='').sum()
    print(f'{c}, NaN: {n_nan} blank: {n_blank}')

In [ ]:
print(off_selected_columns[nutr7].dtypes)
#must have columns
required =['product_name','ingredients_text','image_url'] + nutr7

off_no_missing = off_selected_columns.dropna(subset= required).copy()
print(len(off), 'to', len(off_no_missing))

In [ ]:
rename = {
    'proteins_100g':'protein_g',
    'fat_100g': 'fat_g',
    'carbohydrates_100g':'carb_g',
    'sugars_100g':'sugars_g',
    'fiber_100g':'fiber_g',
    'saturated-fat_100g':'sat_fat_g',
    'sodium_100g': 'sodium_g',
    'energy-kcal_100g':'kcal'
}

off_no_missing = off_no_missing.rename(columns = rename).copy()
print(off_no_missing.head(5))

In [ ]:
macros = ['protein_g','fat_g','carb_g','sugars_g','fiber_g','sat_fat_g','sodium_g']
print(off_no_missing[macros].describe().T.round(2))


In [ ]:
# run some checks to see if the value are within reasoble range
eps = 0.01
checks = {
    'out_of_range': ~((off_no_missing[macros] >= -eps) & (off_no_missing[macros]<=100 +eps)).all(axis=1),
    'sodium_over_40': off_no_missing['sodium_g']> 40,
    'sugar_gt_carb': off_no_missing['sugars_g']> off_no_missing['carb_g'] * 1.05 + 5,
    'satfat_gt_fat': off_no_missing['sat_fat_g']> off_no_missing['fat_g'] * 1.05 +0.5,
    'macros_over_100': (off_no_missing['protein_g']+off_no_missing['fat_g']+off_no_missing['carb_g'])>105,   
}

for name, mask in checks.items():
    print(f'{name:16s} {mask.sum()}')

bad = pd.concat(checks.values(), axis =1).any(axis=1)
print('rows falling any check:', bad.sum(), 'of', len(off_no_missing))
print(off_no_missing.loc[bad, ['product_name'] + macros].head(20))

In [ ]:
off_f = off_no_missing[~bad].copy()
print('after range checks:', len(off_f))
print(off_f[macros].describe().T.round(2))

In [ ]:
# remove any duplicate rows by keeping unique barcode
off_f = off_f.drop_duplicates(subset='code', keep='first')
print('after duplicate codes:', len(off_f))

print(off_f['code'].str.len().describe())
print(off_f.isna().sum())

In [ ]:
# check how long are most of the barcode
L = off_f['code'].str.len()
print(L.value_counts().sort_index())
print((~off_f['code'].str.fullmatch(r'\d+')).sum(), 'codes contain non-digits')

In [ ]:
# check number rows per country
print(off_f['countries_en'].value_counts().head(50))

In [ ]:
# eng = 'Australia|New Zealand|United Kingdom|United States|Ireland|Canada|Singapore'
# is_eng = off_f['countries_en'].str.contains(eng, na= False)
# print('sold in an English speaking country:', is_eng.sum(), 'of', len(off_f))

In [ ]:
has_letters = off_f['ingredients_text'].str.contains(r'[A-Za-z]{3,}', na= False)
off_clean = off_f[has_letters].copy()
print(off_clean.shape)

In [ ]:
off_clean.head(5)

In [ ]:
print(off_clean['kcal'].dtype)
print(off_clean['kcal'].notna().sum(), 'of', len(off_clean))

In [ ]:
p, fat, c, fib = (off_clean[k] for k in ['protein_g','fat_g','carb_g','fiber_g'])
kcal = off_clean['kcal']

calc = 4*p + 4*c + 2*fib + 9*fat
gap_per_fiber = (calc-kcal)/fib

test= (fib >= 1) & kcal.notna() & (kcal >= 0)
say_incl = test& (gap_per_fiber >3)
say_excl = test& (gap_per_fiber <1)

print('tested:', test.sum(), '|say include:', say_incl.sum(), '|say exclude:', say_excl.sum())

In [ ]:
off_clean['carb_excl_fiber_g']= off_clean['carb_g']
off_clean.loc[say_incl, 'carb_excl_fiber_g']=(off_clean.loc[say_incl, 'carb_g']- fib[say_incl])
print(say_incl.sum(), 'rows adjusted')

In [ ]:
print((off_clean["carb_excl_fiber_g"] != off_clean["carb_g"]).sum())

In [ ]:
neg = off_clean["carb_excl_fiber_g"] < 0
print(off_clean.loc[neg, "carb_excl_fiber_g"].describe())

In [ ]:
off_clean["carb_excl_fiber_g"] = off_clean["carb_excl_fiber_g"].clip(lower=0)
print((off_clean["carb_excl_fiber_g"] < 0).sum()) 

In [ ]:
def clean_name(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("'", "").replace(",", "").replace("(", "").replace(")", "")
    return ' '.join(text.split())

In [ ]:
off_clean['name_clean']= off_clean['product_name'].apply(clean_name)
print(off_clean.shape)
print(off_clean.head(10))

In [ ]:
off_clean.to_csv('/kaggle/working/off_filtered_all_countries.csv', index=False)